# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Load the dataset from GitHub
url = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv'
df = pd.read_csv(url)

# Inspect basic shape
print(f"Initial shape: {df.shape}")

# 2. Basic cleanup of missing values before split
# Check for missing values
missing_cols = df.isnull().sum()
print("Missing values per column prior to handling:")
print(missing_cols[missing_cols > 0])

# Impute or drop missing values
# Usually 'country', 'children', 'agent', and 'company' might have missing values
if 'children' in df.columns:
    df['children'] = df['children'].fillna(0)
if 'country' in df.columns:
    df['country'] = df['country'].fillna('Unknown')
if 'agent' in df.columns:
    df['agent'] = df['agent'].fillna('No Agent')
if 'company' in df.columns:
    df['company'] = df['company'].fillna('No Company')

# Drop reservation_status because it perfectly correlates with the target is_canceled
if 'reservation_status' in df.columns:
    df = df.drop(columns=['reservation_status'])

# Drop other highly leakable or non-predictive variables if present
columns_to_drop = ['reservation_status_date']
for col in columns_to_drop:
    if col in df.columns:
        df = df.drop(columns=[col])

# 3. Create X (features) and y (target)
y = df['is_canceled']
X = df.drop(columns=['is_canceled'])

# 4. Split the data into training and test sets (70/30) FIRST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# 5. Encode categorical variables on the split datasets
# Make copies to avoid SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()

categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Apply Label Encoding (or One-Hot Encoding) fit on Train and transform on Test
for col in categorical_cols:
    le = LabelEncoder()
    # Convert columns to string to prevent mixing types error
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

    # Fit on training data
    le.fit(X_train[col])

    # Handle unseen categories in test set by mapping them to an 'Unknown' class
    # if they were not present in the training set
    test_categories = set(X_test[col].unique())
    train_categories = set(le.classes_)
    unseen = test_categories - train_categories

    if unseen:
        # Re-fit label encoder with 'Unknown' category added
        new_classes = np.append(le.classes_, 'Unknown')
        le.classes_ = new_classes
        X_test[col] = X_test[col].apply(lambda x: x if x in train_categories else 'Unknown')

    X_train[col] = le.transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

print(f"\nData split completed:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Initial shape: (9404, 32)
Missing values per column prior to handling:
country     266
agent      2335
company    8393
dtype: int64

Data split completed:
X_train shape: (6582, 29)
X_test shape: (2822, 29)


### ✍️ Your Response: 🔧
1. **How many total rows and columns are in the dataset?**
  
This dataset initially has 9,404 rows and 32 columns.

2. **What types of features (categorical, numerical) are included?**

The numerical features that are included are those such as lead time, arrical date year, stays in weekend nights, stays in week nights, adults, children, etc. The categorical features includes those like hotel, arrival date month, meal, country, market segment, reserved room type, etc.
  


3. **What steps did you take to clean or prepare the data?**

I first imputed the missing values by filling in missing entires in children with the number 0, the missing text values with unknown, the missing agents with no agent, etc. I then dropped some columns that created data leakage, for example reservation status got dropped because of the is canceled variable. I then conducted a train/test split where the x and y variables were seperated. I also converted some categorical variables into numerical values when necessary.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [2]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize the Naïve Bayes Classifier
nb_model = GaussianNB()

# 2. Fit the model on the training data
nb_model.fit(X_train, y_train)

# 3. Predict on the test data
y_pred_nb = nb_model.predict(X_test)

# 4. Print evaluation results
print("=== Naïve Bayes Classification Report ===")
print(classification_report(y_test, y_pred_nb))

print("=== Naïve Bayes Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_nb))

=== Naïve Bayes Classification Report ===
              precision    recall  f1-score   support

           0       0.95      0.79      0.86      2121
           1       0.58      0.87      0.69       701

    accuracy                           0.81      2822
   macro avg       0.76      0.83      0.78      2822
weighted avg       0.86      0.81      0.82      2822

=== Naïve Bayes Confusion Matrix ===
[[1677  444]
 [  94  607]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. The model got an overall accuracy of 81% for predicting cancellations and a high recall of 87%. There is a pretty low precision level though of 58%. The recall or F1 metrics are the best to judge performance because not being able to identify a guest who will cancel will cause over-preparing issues for individuals who will ultimately no-show.


2. This model can be useful for the hotel as a real-time screening tool. The Naive Bayes tool is very efficient and opperates with low latency, this means that it can flag a newly created booking as a high risk cancellation immediately when the information is entered. This will help the managers to figure out a booking schedule, without having to worry about impacting the booking platform metrics.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# 1. Scale the data using StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training set and transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Train an SVM classifier (use linear kernel)
# Setting random_state for reproducibility
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_scaled, y_train)

# 3. Make predictions on the scaled test data
y_pred_svm = svm_model.predict(X_test_scaled)

# 4. Evaluate with classification metrics
print("=== SVM (Linear Kernel) Classification Report ===")
print(classification_report(y_test, y_pred_svm))

print("=== SVM (Linear Kernel) Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_svm))

=== SVM (Linear Kernel) Classification Report ===
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      2121
           1       0.88      0.55      0.68       701

    accuracy                           0.87      2822
   macro avg       0.87      0.76      0.80      2822
weighted avg       0.87      0.87      0.86      2822

=== SVM (Linear Kernel) Confusion Matrix ===
[[2067   54]
 [ 316  385]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. The SVM Model performs well with an overall accuracy of 87%. When predicting cancellations, SVM has a high precision of 88% but a low recall of 55%. Precision is the best metric to judge performance because we do not want to waste resources on guests who are going to cancel anyways, so this helps us lessen the losses on our part.


2. SVM with non-linear kernels works well when there is a complex, multi-dimensional dataset to work with. SVM provides better insights than the simpler models because it allows us to capture very subtle differences in behavior that simple models might fail to detect.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [4]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize the MLPClassifier with a simple architecture (2 hidden layers: 64 and 32 nodes)
# Use early_stopping=True and validation_fraction=0.15 to use a true internal validation split
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15
)

# 2. Fit the model using our scaled training data
# Neural networks are sensitive to feature scaling, so we use X_train_scaled
mlp_model.fit(X_train_scaled, y_train)

# 3. Predict on the scaled test data
y_pred_mlp = mlp_model.predict(X_test_scaled)

# 4. Print evaluation results
print("=== Neural Network (MLP) Classification Report ===")
print(classification_report(y_test, y_pred_mlp))

print("=== Neural Network (MLP) Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_mlp))

=== Neural Network (MLP) Classification Report ===
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      2121
           1       0.83      0.81      0.82       701

    accuracy                           0.91      2822
   macro avg       0.88      0.88      0.88      2822
weighted avg       0.91      0.91      0.91      2822

=== Neural Network (MLP) Confusion Matrix ===
[[2001  120]
 [ 131  570]]


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. The Neural Network Model compares to the others because it performs way better than the rest. It has the highest overall accuracy of 91%. It also has 83% precision and 81% recall, which means it is able to capture a large amount of predicted cancellations.


2. I think the business would be comfortable using this, but only to an extent. This is because this model has a high accuracy and also a high operational efficiency. If this is used for background operations only, then the business will definitely want to use it to get all the backend calculations done. They will only use this to an extent because they will not use it for direct customer-facing policies. This will be saved for managers because they can control the relationships with their customers still. Otherwise a more simple model could be used for preventing any brand damage and helping with customer complaints, where we just need a decision tree model with simple categorical data and binning.



## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [5]:
from sklearn.metrics import accuracy_score

# Calculate accuracy scores using actual and predicted values from earlier runs
accuracy_nb = accuracy_score(y_test, y_pred_nb)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)

# Print accuracy results
print("=== Model Accuracy Comparison ===")
print(f"Naïve Bayes Accuracy : {accuracy_nb:.4f} ({accuracy_nb * 100:.2f}%)")
print(f"Linear SVM Accuracy  : {accuracy_svm:.4f} ({accuracy_svm * 100:.2f}%)")
print(f"Neural Network (MLP) : {accuracy_mlp:.4f} ({accuracy_mlp * 100:.2f}%)")
print("=================================\n")

# Identify and print the best-performing model
accuracies = {
    "Naïve Bayes": accuracy_nb,
    "Support Vector Machine (Linear SVM)": accuracy_svm,
    "Neural Network (MLP)": accuracy_mlp
}
best_model = max(accuracies, key=accuracies.get)

print(f"Summary: The best performing model is the {best_model} with an accuracy of {accuracies[best_model]*100:.2f}%.")

=== Model Accuracy Comparison ===
Naïve Bayes Accuracy : 0.8094 (80.94%)
Linear SVM Accuracy  : 0.8689 (86.89%)
Neural Network (MLP) : 0.9111 (91.11%)

Summary: The best performing model is the Neural Network (MLP) with an accuracy of 91.11%.


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. In regards to those metrics, the Neural Network model had the best overall metrics, but had the best accuracy of 91.11%. The training time was the best for the Naive Bayes model by being almost instantaneous. Naive Bayes and Linear SVM are both very interpretable. Naive Bayes is probably the easiest to use because it requires no hyperparameter tuning. The other models needs scaling to perform optimally, which Naive Bayes does not need.
  

2. I would recommend deploying the neural network model even though it is less self-explanatory on the face. When looking at hospitality management, having a 4% accuracy prediction difference will add up over the month and save a lot of money for the company by successfully predicting cancellations. This will directly translate into better booking strategies. Being less interpretable is a very small tradeoff considering the advantages that the Neural Network model gives for better accuracy.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧

1. I recommend implementing the Neural Network Model because it helps to solve the important business problem of booking unpredictability. This directly impacts the way that the company staffs, stocks inventory, and plans for extra expenses. Some risks and limitations of this is the "black-box" style of the Neural Network model, which is very difficult in itself to explain to staff. To improve results in the future, we should collect more data including customer review scores, room pricing fluctuations, and historical booking changes.


2. This assignment directly relates to my customized learning outcome because now I understand better what different models can help with specifically. There are now examples of the Neural Network model, the Naive Bayes, and SVM Model that I see the differences on how they use a dataset and develop it. I can use this when talking about developing the ski industry to understand what people might buy which skis, where I can develop a model now for this.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [6]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] Converting notebook assignment_12_bayes_svm_neural.ipynb to html
[NbConvertApp] Writing 334688 bytes to assignment_12_bayes_svm_neural.html
